# Demo 04 — Learning Taxonomies

**Class 01 · D10** (slides: `slides/01_foundations.md`)

One dataset (handwritten digits, 8×8 pixels, `datasets/digits.csv.zst`), four ways of learning from it, a regression example and a small reinforcement learning world.
What changes is **where the learning signal comes from**.

| Setting | Signal | Class |
|---|---|---|
| Supervised | human labels for every sample | 03-07 |
| Unsupervised | the structure of the data alone | 09-10 |
| Semi-supervised | a few labels + many unlabelled samples | 11-12 |
| Self-supervised | labels created from the data itself | 11-12 |
| Reinforcement | rewards obtained by acting in an environment | 13 |

Every cell that prints a number of a slide starts with `slide: <frame title>`; plots go to `assets/figures/plots/01/`.
Plots saved for the slides: `d10_regression`, `d10_accuracy` (label-fraction sweep), `d10_policy`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from matplotlib.patches import Rectangle
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier

from faa import plotlib

plotlib.setup(module="01")
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 42
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(160)


def show(table: pl.DataFrame, digits: int = 4) -> None:
    """Print a table with rounded floats."""
    print(table.with_columns(pl.col(pl.Float64).round(digits)))


def slide(title: str) -> None:
    print(f"--- slide: {title} ---")


def make_classifier():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))


def show_digits(images, titles, size=(4.25, 0.6)):
    """A row of 8 x 8 images (not saved: only for the reader of the notebook)."""
    fg, axs = plt.subplots(1, len(images), figsize=size)
    for ax, img, title in zip(axs, images, titles):
        ax.imshow(img.reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        ax.set(title=title, xticks=[], yticks=[])
        ax.grid(False)
    display(fg)
    plt.close(fg)


digits = pl.read_csv("../../datasets/digits.csv.zst")
X = digits.drop("target").to_numpy() / 16.0
y = digits["target"].to_numpy()
split = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
X_train, X_test, y_train, y_test = (np.asarray(a) for a in split)
print(f"{len(X_train)} training images, {len(X_test)} test images, {X.shape[1]} pixels each")
show_digits([X_train[y_train == d][0] for d in range(10)], [str(d) for d in range(10)])

## 1. Supervised learning — every image has a label

Learn $f: \text{image} \rightarrow \text{digit}$ from $(x_i, y_i)$ pairs.

### Classification: digits (slide: Classification and Regression)

In [ ]:
# slide: Supervised Learning; Classification and Regression
slide("Supervised Learning; Classification and Regression (classification)")
supervised = make_classifier().fit(X_train, y_train)
acc_supervised = supervised.score(X_test, y_test)
print(
    f"Supervised, {len(y_train)} labels: test accuracy = {acc_supervised:.4f} ({len(y_test)} test images, 10 classes, chance = 0.10)"
)

### Regression: miles per gallon from the weight of the car (slide: Classification and Regression; plot `d10_regression`)

The same supervised setting with a **number** as the label (`datasets/auto_mpg.csv.zst`).
The error is reported as the **RMSE** (the square root of the MSE, in miles per gallon, so that it has the unit of the label); everywhere else in Class 01 the "test error" is the MSE.

In [ ]:
# slide: Classification and Regression (regression)
slide("Classification and Regression (regression)")
cars = pl.read_csv("../../datasets/auto_mpg.csv.zst").select("weight", "mpg")
Xc, yc = cars["weight"].to_numpy()[:, None], cars["mpg"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=SEED)
line = LinearRegression().fit(Xc_tr, yc_tr)
curve = make_pipeline(StandardScaler(), PolynomialFeatures(2), LinearRegression()).fit(Xc_tr, yc_tr)
rows = []
for name, m in [("linear", line), ("quadratic", curve)]:
    rows.append({"model": name, "train_rmse": float(np.sqrt(np.mean((yc_tr - m.predict(Xc_tr)) ** 2))),
                 "test_rmse": float(np.sqrt(np.mean((yc_te - m.predict(Xc_te)) ** 2))), "test_R2": m.score(Xc_te, yc_te)})  # fmt: skip
print(f"{len(yc)} cars; mpg from 1 feature: weight (lb); {len(yc_tr)} for training, {len(yc_te)} for testing")
print(
    f"linear fit: mpg = {line.intercept_:.2f} + ({line.coef_[0]:.5f}) x weight, i.e. {line.coef_[0] * 1000:.2f} mpg per 1000 lb"
)
show(pl.DataFrame(rows), 3)
print(f"test RMSE (root of the MSE, in mpg): linear {rows[0]['test_rmse']:.3f}, quadratic {rows[1]['test_rmse']:.3f}")

In [ ]:
# plot d10_regression
fg, ax = plotlib.fig("col", 1.7)
wg = np.linspace(Xc.min(), Xc.max(), 200)[:, None]
ax.scatter(Xc_tr, yc_tr, s=5, color=GREY, alpha=0.7, label="training cars")
ax.plot(wg, line.predict(wg), color=BLUE, lw=1.5, label="linear")
ax.plot(wg, curve.predict(wg), color=ORANGE, lw=1.5, label="quadratic")
ax.set(xlabel="weight (lb)", ylabel="miles per gallon")
ax.set_title(f"test RMSE (mpg): linear {rows[0]['test_rmse']:.2f},\nquadratic {rows[1]['test_rmse']:.2f}", fontsize=7.5)
ax.legend(loc="upper right", borderaxespad=0.1, handletextpad=0.3, fontsize=7)
plotlib.save(fg, "d10_regression")

## 2. Unsupervised learning — no labels at all

Ask $k$-means to group the images into 10 clusters. It never sees a label; we use the labels **afterwards** to check the result.

In [ ]:
# slide: Unsupervised Learning and Its Tasks
slide("Unsupervised Learning and Its Tasks")
kmeans = KMeans(n_clusters=10, n_init=10, random_state=SEED).fit(X_train)  # pyright: ignore[reportArgumentType]
print(
    f"agreement between clusters and true digits (adjusted Rand index, 0 = random, 1 = perfect): {adjusted_rand_score(y_train, kmeans.labels_):.3f}"
)
sizes = np.bincount(np.asarray(kmeans.labels_))
print(f"cluster sizes: {sizes.tolist()}")
show_digits(list(kmeans.cluster_centers_), [f"c{c}" for c in range(10)])
X_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_train)
fg, axs = plt.subplots(1, 2, figsize=(4.25, 1.8), sharex=True, sharey=True)
axs[0].scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans.labels_, cmap="tab10", s=3)
axs[0].set_title("what k-means found (no labels)", fontsize=7.5)
axs[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap="tab10", s=3)
axs[1].set_title("true digits (hidden)", fontsize=7.5)
display(fg)
plt.close(fg)

## 3. Semi-supervised learning — only a few labels

Keep the labels of a fraction of the images of each digit and hide the rest (label $-1$). Three learners: supervised on the few labels only, **self-training** (pseudo-labels with confidence 0.9) and **label spreading** (a similarity graph).

### 10% of the labels (slides: Semi-Supervised Learning and the Cluster Assumption)

In [ ]:
# slide: Semi-Supervised Learning and the Cluster Assumption


def labelled_subset(fraction, rng):
    """Indices of `fraction` of the training images of each digit (at least one per digit)."""
    return np.concatenate(
        [
            rng.choice(idx, max(1, round(fraction * len(idx))), replace=False)
            for idx in (np.flatnonzero(y_train == d) for d in range(10))
        ]
    )


def semi_supervised_scores(fraction, rng):
    labelled = labelled_subset(fraction, rng)
    y_partial = np.full_like(y_train, -1)
    y_partial[labelled] = y_train[labelled]
    return {
        "labels": len(labelled),
        "supervised_only": make_classifier().fit(X_train[labelled], y_train[labelled]).score(X_test, y_test),
        "self_training": SelfTrainingClassifier(make_classifier(), threshold=0.9)
        .fit(X_train, y_partial)
        .score(X_test, y_test),
        "label_spreading": LabelSpreading(kernel="knn", n_neighbors=7).fit(X_train, y_partial).score(X_test, y_test),
    }


slide("Semi-Supervised Learning (10% of the labels)")
res = semi_supervised_scores(0.10, np.random.default_rng(SEED))
print(f"labelled: {res['labels']} of {len(y_train)}, unlabelled: {len(y_train) - res['labels']}")
print(f"{'method':50s}{'test accuracy':>14s}")
print(f"{'supervised, ' + str(res['labels']) + ' labels only':50s}{res['supervised_only']:14.3f}")
print(f"{'self-training (pseudo-labelling), ' + str(res['labels']) + ' labels':50s}{res['self_training']:14.3f}")
print(f"{'label spreading (similarity graph), ' + str(res['labels']) + ' labels':50s}{res['label_spreading']:14.3f}")
print(f"{'supervised, all ' + str(len(y_train)) + ' labels (reference)':50s}{acc_supervised:14.3f}")

### Sweep: how many labels are needed? (plot `d10_accuracy`)

One random choice of the labelled images per fraction; the 10% row is the run of the previous cell (the slide quotes it and the plot shows the same point).

In [ ]:
fractions = (0.01, 0.02, 0.05, 0.1, 0.2, 0.5)
rows = []
for frac in fractions:
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore", RuntimeWarning
        )  # label spreading divides by zero for an image far from every labelled one
        run = semi_supervised_scores(frac, np.random.default_rng(SEED))
    rows.append(
        {
            "label_fraction": frac,
            "labels": run["labels"],
            **{k: float(run[k]) for k in ("supervised_only", "self_training", "label_spreading")},
        }
    )
sweep10 = pl.DataFrame(rows)
show(sweep10, 3)
print(f"reference: supervised with all {len(y_train)} labels = {acc_supervised:.3f}")

In [ ]:
# plot d10_accuracy
fg, ax = plotlib.fig("full", 1.8)
for col, colour, mk, lab in [("supervised_only", GREY, "o", "supervised, labelled images only"), ("self_training", BLUE, "s", "self-training"),
                             ("label_spreading", ORANGE, "^", "label spreading")]:  # fmt: skip
    ax.plot(sweep10["label_fraction"] * 100, sweep10[col], color=colour, marker=mk, ms=3, label=lab)
ax.axhline(acc_supervised, color="k", ls=":", lw=1)
ax.annotate("all 1257 labels", (1, acc_supervised), (0, -9), textcoords="offset points", ha="left", fontsize=7.5)
ax.set(
    xscale="log", xlabel="labelled images (% of the training set, log scale)", ylabel="test accuracy", ylim=(0.2, 1.0)
)
ax.set_xticks([1, 2, 5, 10, 20, 50], ["1", "2", "5", "10", "20", "50"])
ax.minorticks_off()
ax.legend(loc="lower right", borderaxespad=0.1, handletextpad=0.3)
plotlib.save(fg, "d10_accuracy")

Unlabelled data is cheap, and the *structure* it reveals (similar images are close together) lets a few labels go much further.
The gain is largest when labels are scarcest and shrinks when there are plenty.
**Failure case:** with 10 labels (1%) self-training is *worse* than the supervised model alone: its first pseudo-labels are wrong and it learns from its own mistakes;
label spreading uses the geometry of all the images and does not.

## 4. Self-supervised learning — the data labels itself

**Pretext task:** hide the bottom half of each image and predict it from the top half. The "labels" (the hidden pixels) come for free, so no human annotation is needed.
The same idea, at a much larger scale, trains autoencoders, language models (predict the masked or next word) and JEPA models.

In [ ]:
# slide: Self-Supervised Learning and Pretext Tasks
slide("Self-Supervised Learning and Pretext Tasks")
top, bottom = X_train[:, :32], X_train[:, 32:]
inpainter = Ridge(alpha=1.0).fit(top, bottom)
predicted_bottom = inpainter.predict(X_test[:, :32])
mse_model = float(np.mean((predicted_bottom - X_test[:, 32:]) ** 2))
mse_mean = float(np.mean((bottom.mean(axis=0) - X_test[:, 32:]) ** 2))
print(
    f"pretext task: predict the 32 bottom pixels of {len(X_test)} test images from the 32 top pixels, no digit label used"
)
print(
    f"MSE of the model = {mse_model:.4f}; MSE of always predicting the average bottom half = {mse_mean:.4f} ({mse_model / mse_mean:.2f} of it)"
)
completed = np.concatenate([X_test[:8, :32], predicted_bottom[:8]], axis=1)
masked = X_test[:8].copy()
masked[:, 32:] = 0.5
show_digits(list(X_test[:8]), ["original"] * 8)
show_digits(list(masked), ["masked"] * 8)
show_digits(list(completed), ["predicted"] * 8)

To predict the missing half, the model must capture how digits are **shaped**. That knowledge is a useful representation for later tasks.

## 5. Reinforcement learning — learning from rewards

There is no dataset. An **agent** in a **state** $s_t$ chooses an **action** $a_t$ and receives a **reward** $R_{t+1}$ and the next state $s_{t+1}$.

| symbol | meaning | here |
|---|---|---|
| $s_t$ | state at time $t$ | a cell of a 5×5 grid |
| $a_t$ | action at time $t$ | up, down, left, right |
| $R_{t+1}$ | reward after the action | goal +1, pit -1, every other step -0.01 |

**Q-learning** keeps a table $Q(s, a)$ ("how good is $a$ in $s$") and nudges it after each step:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \big[\, R_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \,\big]$$

With probability $\varepsilon$ the agent **explores** (random action), otherwise it **exploits** its best known action.

### The grid world (slides: Reinforcement Learning; plot `d10_policy`)

In [ ]:
SIZE, START, GOAL = 5, (0, 0), (4, 4)
PITS = {(1, 1), (2, 3), (3, 1)}
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right as (row, column) steps


def step(state, action, step_cost=-0.01):
    """Next state, reward R_{t+1} and whether the episode ended."""
    r, c = state
    dr, dc = MOVES[action]
    nxt = (min(max(r + dr, 0), SIZE - 1), min(max(c + dc, 0), SIZE - 1))
    if nxt == GOAL:
        return nxt, 1.0, True
    if nxt in PITS:
        return nxt, -1.0, True
    return nxt, step_cost, False


def q_learning(rng, episodes=500, alpha=0.5, gamma=0.95, epsilon=0.2, step_cost=-0.01):
    """Tabular Q-learning; returns the table and the return of every episode."""
    q_table = np.zeros((SIZE, SIZE, 4))
    returns = []
    for _ in range(episodes):
        state, total, done, steps = START, 0.0, False, 0
        while not done and steps < 100:
            action = int(rng.integers(4)) if rng.uniform() < epsilon else int(np.argmax(q_table[state]))
            nxt, reward, done = step(state, action, step_cost)
            target = reward if done else reward + gamma * q_table[nxt].max()
            q_table[state][action] += alpha * (target - q_table[state][action])
            state, total, steps = nxt, total + reward, steps + 1
        returns.append(total)
    return q_table, np.array(returns)


# slide: Reinforcement Learning
slide("Reinforcement Learning")
Q, returns = q_learning(np.random.default_rng(SEED))
print("500 episodes, alpha = 0.5, gamma = 0.95, epsilon = 0.2")
print(f"average return, first 50 episodes: {returns[:50].mean():+.3f}")
print(f"average return, last 50 episodes : {returns[-50:].mean():+.3f}")
print(f"best possible return of an episode: 8 steps to the goal, 7 at -0.01 and +1 = {1 - 7 * 0.01:+.2f}")

In [ ]:
# plot d10_policy: left the return of every episode (moving average), right the learned policy
fg, axs = plotlib.fig("full", 1.9, ncols=2, gridspec_kw={"width_ratios": [1.15, 1]})
window = 20
ax = axs[0]
ax.plot(np.convolve(returns, np.ones(window) / window, mode="valid"), color=BLUE)
ax.set(xlabel="episode", ylabel=f"return (average of {window})")
ax = axs[1]
ax.set(xlim=(-0.5, SIZE - 0.5), ylim=(SIZE - 0.5, -0.5), xticks=[], yticks=[], aspect="equal")
ax.grid(False)
for r in range(SIZE):
    for c in range(SIZE):
        if (r, c) in PITS or (r, c) == GOAL:
            ax.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, color=RED if (r, c) in PITS else GREEN, alpha=0.7))
            ax.text(c, r, "pit" if (r, c) in PITS else "goal", ha="center", va="center", fontsize=7.5)
        else:
            dr, dc = MOVES[int(np.argmax(Q[r, c]))]
            ax.annotate(
                "",
                (c + 0.3 * dc, r + 0.3 * dr),
                (c - 0.3 * dc, r - 0.3 * dr),
                arrowprops={"arrowstyle": "->", "color": "k", "lw": 1.2},
            )
for k in range(SIZE + 1):
    ax.axhline(k - 0.5, color=GREY, lw=0.5)
    ax.axvline(k - 0.5, color=GREY, lw=0.5)
ax.set_title("learned policy: best action per cell", fontsize=7.5)
plotlib.save(fg, "d10_policy")

### Sweep: exploration (no slide)

$\varepsilon$ is the probability of a random action; the step cost is the reward of an ordinary step. 5 seeds per row: average return of the last 50 episodes (it includes the random moves),
the worst seed, and the share of runs that reached the goal at least once in the last 50 episodes.
**Failure case:** with no step cost and $\varepsilon = 0$ the agent only exploits a table that starts at 0 and repeats whatever it tried first.

In [ ]:
rows = []
for eps, cost in [(0.0, -0.01), (0.05, -0.01), (0.2, -0.01), (0.5, -0.01), (0.0, 0.0), (0.05, 0.0), (0.2, 0.0)]:
    last, reached = [], []
    for s in range(5):
        _, ret = q_learning(np.random.default_rng(SEED + s), epsilon=eps, step_cost=cost)
        last.append(ret[-50:].mean())
        reached.append(float(np.any(ret[-50:] > 0.5)))
    rows.append(
        {
            "epsilon": eps,
            "step_cost": cost,
            "return_last_50": float(np.mean(last)),
            "worst_seed": float(np.min(last)),
            "share_reaching_goal": float(np.mean(reached)),
        }
    )
show(pl.DataFrame(rows), 3)

Nobody told the agent which moves are correct. It discovered a path around the pits **by trial and error**, guided only by rewards.
Reinforcement learning is expanded in Class 13.